# Workshop: Building and Modifying Neural Networks with PyTorch and Hugging Face
**Duration:** about 105 minutes.

In Lab 02_03 and Workshop 02_04 we used pre-trained networks as black boxes and put a logistic regression on top.
Today we open the box:
- build small PyTorch models ourselves, from logistic regression to a model with learnable layer weights,
- adapt them to **new tasks** by changing only the output layer, the loss and the metric,
- see that a Hugging Face model is just another `nn.Module`: we inspect it, replace its head, freeze it, train it with *our* training loop and use parts of it inside our own model,
- train the same model with the Hugging Face `Trainer`, the training loop that most fine-tuning scripts use,
- use the Hugging Face Hub API to find, download, save and reload models.

The guiding idea: **most things stay the same**.
Data loading, the training loop and the evaluation protocol do not care whether the model is a single linear layer or WavLM with 95 million parameters, or whether the task is classification or regression.

**Data:** the PARLO `pataka` recordings (10 s of "pataka" repetitions), loaded with `parlo-loader`, with the participant-level folds from Workshop 02_04.
`pataka` is short, so even the end-to-end parts run on a laptop.

> **Legal reminder:** the PARLO data and everything derived from it (features, trained models, notebook outputs) must stay on your machine.
> Never call `push_to_hub` with a model trained on PARLO.

# Part 0: Setup (5 min)
As in Workshop 02_04, set `PARLO_ROOT` to the corpus folder (or edit the fallback path).

In [ ]:
import os
import inspect
from functools import partial
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from IPython.display import display
from sklearn.metrics import balanced_accuracy_score, mean_absolute_error

from parlo import Corpus
from parlo.torch import ParloDataset
import joblib

SR = 16000
# features are cached outside the repo, in the same folder as in Workshop 02_04 (override with SLP_CACHE)
CACHE_DIR = Path(os.environ.get('SLP_CACHE', Path.home() / '.cache' / 'slp_labs'))


def cached(name, fn):
    """Compute fn() once and store the result in CACHE_DIR/<name>.joblib."""
    path = CACHE_DIR / f'{name}.joblib'
    if path.exists():
        return joblib.load(path)
    result = fn()
    path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(result, path)
    return result

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print('device:', DEVICE)

PARLO_ROOT = Path(os.environ.get('PARLO_ROOT', '~/data/PARLO')).expanduser()
if not (PARLO_ROOT / '0metadata.csv').exists():
    raise FileNotFoundError(f'No PARLO corpus at {PARLO_ROOT}: set PARLO_ROOT or edit the fallback path above')

TASK = 'pataka'
WAVLM = 'microsoft/wavlm-base-plus'

corpus = Corpus(PARLO_ROOT)
FOLDS = corpus.kfold(folds=5, seed=42)        # the same participant folds as in Workshop 02_04

# Part 1: From the PARLO Loader to PyTorch Batches (10 min)
PyTorch trains on **batches**.
Two classes do the work:
- a `Dataset` knows how many examples there are and returns example number `i`,
- a `DataLoader` shuffles the indices, fetches a batch of examples and merges them with a **collate function** into tensors.

`parlo.torch.ParloDataset` turns any PARLO query into a `Dataset` whose examples are table rows (dictionaries).
It never decodes audio: what a row becomes (features, audio, text) is decided by our collate function.

In [ ]:
targets = corpus.targets()          # HC=0, MCI=1, AD=2
CLASSES = list(targets.mapping)

query = (targets.expand('recordings')                      # recordings joined with the participant target
         .filter(pl.col('task') == TASK)
         .select('participant_id', 'recording_id', 'diagnosis', 'target')
         .sort('participant_id'))

train_set = ParloDataset(FOLDS[0].apply(query, 'train'))   # fold 0 for development
test_set = ParloDataset(FOLDS[0].apply(query, 'test'))
print(len(train_set), 'training and', len(test_set), 'test recordings')
train_set[0]

## Features
For Parts 1 to 4 we use the frozen WavLM features from Workshop 02_04: the mean-pooled hidden states of **all 13 layers** (13 × 768 numbers per recording).
If you ran Workshop 02_04, they are loaded from the cache; otherwise the extraction takes a few minutes.

In [ ]:
from transformers import AutoFeatureExtractor, AutoModel

CHUNK = 10 * SR


def chunks(samples):
    starts = range(0, max(len(samples) - SR, 1), CHUNK)        # drop a final chunk shorter than 1 s
    return [samples[s:s + CHUNK] for s in starts]


@torch.inference_mode()
def wavlm_features(task):
    # identical to Workshop 02_04, so both notebooks share the cache
    extractor = AutoFeatureExtractor.from_pretrained(WAVLM)
    model = AutoModel.from_pretrained(WAVLM).to(DEVICE).eval()
    rows = corpus.filter(task=task).recordings.sort('participant_id').collect()
    out = []
    for row in tqdm(rows.iter_rows(named=True), total=rows.height, desc=f'WavLM {task}'):
        samples = corpus.load_audio(row, mono=True, sample_rate=SR).samples
        pooled, weights = [], []
        for chunk in chunks(samples):
            inputs = extractor(chunk, sampling_rate=SR, return_tensors='pt').to(DEVICE)
            hidden = model(**inputs, output_hidden_states=True).hidden_states
            pooled.append(torch.stack([h[0].mean(dim=0) for h in hidden]).cpu().numpy())
            weights.append(len(chunk))
        out.append(np.average(pooled, axis=0, weights=weights))
    return rows['participant_id'].to_list(), np.stack(out)


ids, H = cached(f'parlo_wavlm_{TASK}', lambda: wavlm_features(TASK))
FEATS = dict(zip(ids, H.astype(np.float32)))
print(H.shape, '= (recordings, layers, dimensions)')

The collate function turns a list of rows into two tensors: the features `x` and the targets `y`.
The `dtype` of the targets is the first thing that depends on the task: class indices are integers (`long`), regression targets are floats.

In [ ]:
def rows_of(dataset):
    return [dataset[i] for i in range(len(dataset))]


def collate(rows, dtype=torch.long):
    x = torch.tensor(np.stack([FEATS[r['participant_id']] for r in rows]))   # (batch, 13, 768)
    y = torch.tensor([r['target'] for r in rows], dtype=dtype)              # (batch,)
    return x, y


train_loader = DataLoader(train_set, batch_size=32, shuffle=True, collate_fn=collate)
test_loader = DataLoader(test_set, batch_size=64, collate_fn=collate)

x, y = next(iter(train_loader))
print('x', tuple(x.shape), x.dtype, '  y', tuple(y.shape), y.dtype, y[:10].tolist())

**Questions**
1. Why does `ParloDataset` return rows instead of tensors? What would you change to train on audio or on transcripts instead of features?
2. Why do we shuffle the training loader but not the test loader?

# Part 2: Your First Models (20 min)
## Building blocks
A PyTorch model is an `nn.Module`: it stores parameters in `__init__` and computes the output in `forward`.
`nn.Sequential` chains modules.
Writing your own module takes only a few lines.
We need two small ones:
- `Standardize` z-normalises the input with the mean and standard deviation of the **training** data.
  They are stored as **buffers**: saved with the model and moved to the GPU with it, but not trained.
- `SelectLayer` picks one WavLM layer from the 13.

In [ ]:
class Standardize(nn.Module):
    def __init__(self, X_train):
        super().__init__()
        self.register_buffer('mean', X_train.mean(dim=0))
        self.register_buffer('std', X_train.std(dim=0) + 1e-6)

    def forward(self, x):
        return (x - self.mean) / self.std


class SelectLayer(nn.Module):
    def __init__(self, layer):
        super().__init__()
        self.layer = layer

    def forward(self, x):            # (batch, layers, dim) -> (batch, dim)
        return x[:, self.layer]


LAYER = 6
X_train = collate(rows_of(train_set))[0]          # all training features, for the normalisation statistics

# logistic regression is a neural network with a single linear layer
logreg = nn.Sequential(Standardize(X_train), SelectLayer(LAYER), nn.Linear(768, 3))
print(logreg)

logits = logreg(x)
print('logits', tuple(logits.shape))
print('probabilities of the first example:', torch.softmax(logits, dim=1)[0].detach().numpy().round(3))

## A small MLP
One hidden layer with a non-linearity and dropout.
`n_out` is the number of outputs: the **only** part of the architecture that depends on the task.
With a single output we flatten `(batch, 1)` to `(batch,)`, so the output has the same shape as the targets.

In [ ]:
def mlp(X_train, n_out, hidden=64, pick=None):
    return nn.Sequential(
        Standardize(X_train),
        pick if pick is not None else SelectLayer(LAYER),
        nn.Linear(768, hidden), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(hidden, n_out),                         # the output layer ("head")
        *([nn.Flatten(0)] if n_out == 1 else []),
    )


def n_params(model, trainable_only=False):
    return sum(p.numel() for p in model.parameters() if p.requires_grad or not trainable_only)

model = mlp(X_train, n_out=3)
print(model)
print('parameters:', n_params(model))

## One training loop for everything
The training loop below is all you need for the rest of the workshop, including the Hugging Face models in Part 6.
In Part 7 we will see that the Hugging Face `Trainer` is the same loop with more options.
It only assumes three things: a model, a loader that yields `(x, y)`, and a loss function `loss_fn(output, y)`.
The optional `forward` argument says how to call the model (Hugging Face models return an object instead of a tensor).

In [ ]:
def call(model, x):
    return model(x)


def train(model, loader, loss_fn, epochs=60, lr=1e-3, weight_decay=1e-2, forward=call, device='cpu'):
    model.to(device)
    if isinstance(loss_fn, nn.Module):
        loss_fn.to(device)                           # e.g. class weights
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=weight_decay)
    history = []
    for _ in range(epochs):
        model.train()
        total = 0.0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            loss = loss_fn(forward(model, x), y)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(y)
        history.append(total / len(loader.dataset))
    return history


@torch.no_grad()
def predict(model, loader, forward=call, device='cpu'):
    model.to(device).eval()
    outputs, targets_ = [], []
    for x, y in loader:
        outputs.append(forward(model, x.to(device)).cpu())
        targets_.append(y)
    return torch.cat(outputs), torch.cat(targets_)


def uar(out, y):
    return balanced_accuracy_score(y, out.argmax(dim=1))

In [ ]:
def class_weights(y, n_classes):
    counts = torch.bincount(y, minlength=n_classes).float()
    return len(y) / (n_classes * counts)              # like class_weight='balanced' in scikit-learn

y_train = collate(rows_of(train_set))[1]
results_fold0 = {}

for name, build in [('logistic regression', lambda: nn.Sequential(Standardize(X_train), SelectLayer(LAYER), nn.Linear(768, 3))),
                    ('MLP', lambda: mlp(X_train, n_out=3))]:
    torch.manual_seed(0)
    model = build()
    history = train(model, train_loader, nn.CrossEntropyLoss(weight=class_weights(y_train, 3)))
    results_fold0[name] = uar(*predict(model, test_loader))
    plt.plot(history, label=name)
    print(f'{name:20s} {n_params(model):7d} parameters   fold 0 test UAR = {results_fold0[name]:.3f}')

plt.xlabel('epoch')
plt.ylabel('training loss')
plt.legend()
plt.show()

**Questions**
1. In which sense is `nn.Linear(768, 3)` + cross-entropy the same as scikit-learn's `LogisticRegression`? What plays the role of the regularisation parameter `C`?
2. The training loss keeps decreasing. Does that mean the model keeps getting better? What is missing to decide when to stop?
3. Fold 0 alone has about 40 test participants. How much would you trust a difference of 0.05 UAR between the two models?

**Your turn:** change `hidden`, the dropout or `LAYER` and rerun the cell.

# Part 3: New Task, New Head, New Loss (20 min)
Now we keep the data pipeline, the model body and the training loop, and solve **different tasks**.
The loader offers several targets: `targets('hc_vs_impaired')` (HC = 0, MCI and AD = 1) and `targets('mmse')` (the Mini-Mental State Examination score, 0 to 30, a regression target).

What changes per task is collected in a small **recipe**:

| | output layer | loss | target dtype | prediction | metric |
|---|---|---|---|---|---|
| 3 classes | 3 logits | cross-entropy | `long` | argmax | UAR |
| 2 classes | 2 logits | cross-entropy | `long` | argmax | UAR |
| 2 classes | 1 logit | binary cross-entropy (sigmoid) | `float` | logit > 0 | UAR |
| regression | 1 value | mean squared error | `float` | the value | MAE |

`nn.CrossEntropyLoss` and `nn.BCEWithLogitsLoss` expect **logits** (no softmax/sigmoid in the model), because they compute the log-probabilities in a numerically stable way.

In [ ]:
def last_linear(model):
    return [m for m in model.modules() if isinstance(m, nn.Linear)][-1]


def start_at_mean(model, y_train):
    # regression: start from the "predict the training mean" baseline instead of 0
    last_linear(model).bias.data.fill_(y_train.mean().item())


RECIPES = {
    'diagnosis, 3 logits + CE': dict(
        targets=targets, n_out=3, dtype=torch.long, metric=uar, metric_name='UAR',
        loss=lambda y: nn.CrossEntropyLoss(weight=class_weights(y, 3))),
    'HC vs. impaired, 2 logits + CE': dict(
        targets=corpus.targets('hc_vs_impaired'), n_out=2, dtype=torch.long, metric=uar, metric_name='UAR',
        loss=lambda y: nn.CrossEntropyLoss(weight=class_weights(y, 2))),
    'HC vs. impaired, 1 logit + BCE': dict(
        targets=corpus.targets('hc_vs_impaired'), n_out=1, dtype=torch.float32, metric_name='UAR',
        metric=lambda out, y: balanced_accuracy_score(y, out > 0),
        loss=lambda y: nn.BCEWithLogitsLoss(pos_weight=(y == 0).sum() / (y == 1).sum())),
    'MMSE, 1 value + MSE': dict(
        targets=corpus.targets('mmse'), n_out=1, dtype=torch.float32, init=start_at_mean, metric_name='MAE',
        metric=lambda out, y: mean_absolute_error(y, out),
        loss=lambda y: nn.MSELoss()),
}

The cross-validation function is the same for all recipes.
It uses the participant folds from `corpus.kfold()` for every target (participants without an MMSE value are simply not in the MMSE query).

In [ ]:
def make_loaders(target_view, fold, dtype):
    q = target_view.expand('recordings').filter(pl.col('task') == TASK).select('participant_id', 'target').sort('participant_id')
    col = partial(collate, dtype=dtype)
    train_set_, test_set_ = (ParloDataset(FOLDS[fold].apply(q, part)) for part in ('train', 'test'))
    return DataLoader(train_set_, batch_size=32, shuffle=True, collate_fn=col), DataLoader(test_set_, batch_size=64, collate_fn=col)


def run_cv(recipe, build=mlp, epochs=60):
    scores = []
    for fold in range(len(FOLDS)):
        torch.manual_seed(fold)
        train_loader_, test_loader_ = make_loaders(recipe['targets'], fold, recipe['dtype'])
        X_tr, y_tr = collate(rows_of(train_loader_.dataset), recipe['dtype'])
        model = build(X_tr, recipe['n_out'])
        if 'init' in recipe:
            recipe['init'](model, y_tr)
        train(model, train_loader_, recipe['loss'](y_tr), epochs)
        scores.append(recipe['metric'](*predict(model, test_loader_)))
    return np.mean(scores), np.std(scores)


cv_results = {}
for name, recipe in RECIPES.items():
    cv_results[name] = run_cv(recipe)
    print(f'{name:35s} {recipe["metric_name"]} = {cv_results[name][0]:.3f} ± {cv_results[name][1]:.3f}')

A regression result needs a baseline: how good is a "model" that always predicts the mean MMSE of the training participants?

In [ ]:
mmse = corpus.targets('mmse').participants.select('participant_id', 'diagnosis', 'target').collect().to_pandas().set_index('participant_id')

baseline = []
for f in FOLDS:
    train_mmse = mmse.target.reindex(f.assignments['train']).dropna()
    test_mmse = mmse.target.reindex(f.assignments['test']).dropna()
    baseline.append((test_mmse - train_mmse.mean()).abs().mean())
print(f'MAE when always predicting the training mean: {np.mean(baseline):.3f} ± {np.std(baseline):.3f}')
mmse.groupby('diagnosis').target.describe().round(1)

## Changing only the loss
Everything else fixed, we compare three losses for the three-class task:
- plain cross-entropy (every participant counts the same, so the larger classes dominate),
- class-weighted cross-entropy (every class counts the same, matching the UAR),
- cross-entropy with **label smoothing**: the target is 0.9 for the true class instead of 1, which discourages over-confident predictions.

In [ ]:
LOSSES = {
    'CE': lambda y: nn.CrossEntropyLoss(),
    'class-weighted CE': lambda y: nn.CrossEntropyLoss(weight=class_weights(y, 3)),
    'class-weighted CE + label smoothing 0.1': lambda y: nn.CrossEntropyLoss(weight=class_weights(y, 3), label_smoothing=0.1),
}
for name, loss in LOSSES.items():
    m, s = run_cv(dict(RECIPES['diagnosis, 3 logits + CE'], loss=loss))
    print(f'{name:42s} UAR = {m:.3f} ± {s:.3f}')

## Re-using a trained body for a new task
A trained network can be re-used: keep the body, **replace the head**, and train only the new head (the body is frozen).
This is exactly what happens when you put a new classifier on a pre-trained model, only on a small scale.
We train on diagnosis, then switch to MMSE regression on the same fold.

In [ ]:
torch.manual_seed(0)
model = mlp(X_train, n_out=3)
train(model, train_loader, nn.CrossEntropyLoss(weight=class_weights(y_train, 3)))

model[-1] = nn.Linear(64, 1)          # new output layer for regression ...
model.append(nn.Flatten(0))           # ... with output shape (batch,)
for p in model[:-2].parameters():     # freeze the body
    p.requires_grad_(False)
print(model)
print(f'trainable parameters: {n_params(model, trainable_only=True)} of {n_params(model)}')

mmse_train, mmse_test = make_loaders(corpus.targets('mmse'), 0, torch.float32)
start_at_mean(model, collate(rows_of(mmse_train.dataset), torch.float32)[1])
train(model, mmse_train, nn.MSELoss())
print(f'fold 0 MAE with the frozen diagnosis body: {mean_absolute_error(*predict(model, mmse_test)):.3f}   (baseline fold 0: {baseline[0]:.3f})')

**Questions**
1. Compare the two binary variants (2 logits + CE vs. 1 logit + BCE). Why are they mathematically almost the same model?
2. Why is it a bad idea to put `nn.Softmax` at the end of the model and then use `nn.CrossEntropyLoss`?
3. Is the MMSE model better than the mean-predictor baseline? Why is the MMSE so hard to predict from 10 s of "pataka"?
4. Which loss fits the UAR best, and why?
5. What did you have to change in the code to go from classification to regression? List the lines.

**Your turn:**
- Replace `nn.MSELoss()` with `nn.L1Loss()` (it optimises the MAE directly) or `nn.HuberLoss()`.
- Add a recipe for `corpus.targets(labels=['HC', 'AD'])` (HC vs. AD only).
- Set `TASK = 'pictdes'` in Part 0 and rerun Parts 1 to 3 (the features are cached by Workshop 02_04).

# Part 4: A Module of Your Own: Learnable Layer Weights (10 min)
So far we picked one WavLM layer by hand.
Instead, the model can learn a **weighted sum of all layers**, with one weight per layer (softmax-normalised so they sum to 1).
The weights are an `nn.Parameter`, so the optimiser trains them together with everything else.
This is the standard way to use SSL models in the SUPERB benchmark, and Hugging Face has it built in (Part 6).

In [ ]:
class LayerWeightedSum(nn.Module):
    def __init__(self, n_layers):
        super().__init__()
        self.logits = nn.Parameter(torch.zeros(n_layers))       # start with equal weights

    def weights(self):
        return torch.softmax(self.logits, dim=0)

    def forward(self, x):                                       # (batch, layers, ...) -> (batch, ...)
        w = self.weights().view(1, -1, *[1] * (x.dim() - 2))
        return (w * x).sum(dim=1)


weighted = lambda X, n_out: mlp(X, n_out, pick=LayerWeightedSum(13))
m, s = run_cv(RECIPES['diagnosis, 3 logits + CE'], build=weighted)
print(f'MLP on learned layer mix   UAR = {m:.3f} ± {s:.3f}')
print(f'MLP on layer {LAYER} (Part 3)     UAR = {cv_results["diagnosis, 3 logits + CE"][0]:.3f} ± {cv_results["diagnosis, 3 logits + CE"][1]:.3f}')

torch.manual_seed(0)
model = weighted(X_train, 3)
train(model, train_loader, nn.CrossEntropyLoss(weight=class_weights(y_train, 3)), epochs=200)
plt.bar(range(13), model[1].weights().detach().numpy())
plt.axhline(1 / 13, color='gray', ls=':', label='initial (uniform)')
plt.xlabel('WavLM layer')
plt.ylabel('learned weight')
plt.legend()
plt.show()

**Questions**
1. How many parameters did `LayerWeightedSum` add? Why does this make it a cheap alternative to selecting the layer by cross-validation?
2. Which layers get the largest weights? Compare with the layer-wise probing in Lab 02_03.
3. How far do the weights move away from uniform? Which settings of the training loop (learning rate, weight decay, number of epochs) influence that?

# Part 5: The Hugging Face Hub (5 min)
The [Hugging Face Hub](https://huggingface.co/models) hosts the pre-trained models we have used (WavLM, HuBERT, Whisper, BERT, ...).
`huggingface_hub` is its Python API: search, inspect and download files.
`from_pretrained(name)` does the download for you and caches the files in `~/.cache/huggingface`.

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
from transformers import AutoConfig

api = HfApi()
for m in api.list_models(search='wavlm', sort='downloads', limit=5):
    print(f'{m.id:45s} {m.downloads:>10,} downloads   {m.pipeline_tag}')

info = api.model_info(WAVLM)
print('\nfiles of', info.id, ':', [s.rfilename for s in info.siblings])
print('config file cached at', hf_hub_download(WAVLM, 'config.json'))

config = AutoConfig.from_pretrained(WAVLM)
print(f'\n{config.model_type}: {config.num_hidden_layers} Transformer layers, hidden size {config.hidden_size}')

A Hugging Face repository is just a set of files: a `config.json` (the architecture), the weights and, for audio models, a `preprocessor_config.json` (sampling rate, normalisation).
The `Auto*` classes read the config and build the matching PyTorch model:
- `AutoModel`: the backbone only (hidden states out), as in Lab 02_03,
- `AutoModelForAudioClassification`, `AutoModelForSequenceClassification` (text), `AutoModelForCTC` (ASR), ...: the same backbone plus a task-specific **head**.

**Your turn:** search for `hubert` or `whisper` instead, or look at the files of a text model such as `bert-base-german-cased`.

# Part 6: Hugging Face Models Are `nn.Module`s (20 min)
Now we leave the cached features and train on **audio**.
The collate function changes (it loads audio and runs the feature extractor), the training loop does not.

Training examples are random 4 s crops of the recording (a simple data augmentation), test examples the central 4 s.
All `pataka` recordings of the corpus are loaded into memory once (about 0.6 MB per recording).

In [ ]:
CROP = 4 * SR
extractor = AutoFeatureExtractor.from_pretrained(WAVLM)

AUDIO = {row['participant_id']: corpus.load_audio(row, mono=True, sample_rate=SR).samples
         for row in tqdm(list(query.collect().iter_rows(named=True)), desc='loading audio')}


def crop(samples, random):
    if len(samples) < CROP:
        samples = np.pad(samples, (0, CROP - len(samples)))
    start = np.random.randint(0, len(samples) - CROP + 1) if random else (len(samples) - CROP) // 2
    return samples[start:start + CROP]


def collate_audio(rows, random=False):
    batch = extractor([crop(AUDIO[r['participant_id']], random) for r in rows], sampling_rate=SR, return_tensors='pt')
    return batch['input_values'], torch.tensor([r['target'] for r in rows])


audio_train = DataLoader(train_set, batch_size=8, shuffle=True, collate_fn=partial(collate_audio, random=True))
audio_test = DataLoader(test_set, batch_size=8, collate_fn=collate_audio)
xa, ya = next(iter(audio_train))
print('x', tuple(xa.shape), '= (batch, samples)   y', ya.tolist())

## A ready-made classification model
`AutoModelForAudioClassification` puts a head on WavLM.
We tell it the number of labels and their names, and ask for the learnable weighted layer sum from Part 4 (`use_weighted_layer_sum=True`).
Every keyword argument of `from_pretrained` that matches a config entry overrides the config: we also switch off `layerdrop`, a regularisation that randomly skips whole Transformer layers in training mode (it would also break the layer sum, which expects all 13 layers).
The warning about **newly initialised** weights is expected: the head does not exist in the pre-trained checkpoint, it is random until we train it.

In [ ]:
from transformers import AutoModelForAudioClassification



def hf_classifier():
    return AutoModelForAudioClassification.from_pretrained(
        WAVLM, num_labels=3, label2id=targets.mapping, id2label={i: c for c, i in targets.mapping.items()},
        use_weighted_layer_sum=True,
        layerdrop=0.0)      # no random skipping of Transformer layers during training (only useful for fine-tuning)


hf_model = hf_classifier()

print(type(hf_model).__name__, 'is an nn.Module:', isinstance(hf_model, nn.Module))
for name, child in hf_model.named_children():
    print(f'  {name:15s} {type(child).__name__:25s} {n_params(child):>11,} parameters')
print('  layer_weights  ', tuple(hf_model.layer_weights.shape))

What does the model do with these parts?
We simply read the source code of its `forward` method.

In [ ]:
src = inspect.getsource(type(hf_model).forward)
print(src[src.index('outputs = self.wavlm'):])

You should recognise everything:
- the weighted layer sum is our `LayerWeightedSum` from Part 4 (`self.layer_weights`),
- then a linear `projector` (768 → 256), **mean pooling** over time (as in Lab 02_03), and a linear `classifier` (the output layer),
- if you pass `labels`, it computes the loss for you, and it is **always cross-entropy** for this model.

Let us check that the built-in loss is just `nn.CrossEntropyLoss` on the logits.

In [ ]:
with torch.no_grad():
    out = hf_model(input_values=xa, labels=ya)
print('Hugging Face loss:', out.loss.item())
print('our loss         :', nn.CrossEntropyLoss()(out.logits, ya).item())

Because the built-in loss is fixed (no class weights, no regression for this model), we use **our own loss on the logits**, exactly as before.
The only difference to Part 2 is how the model is called: it takes `input_values=` and returns an output object, so we pass a different `forward` function.

We freeze the 94 M parameters of the backbone and train only the layer weights and the head.
With a frozen backbone the result is equivalent to training on pre-computed features (Part 4), just slower, because WavLM runs again in every epoch.
**Fine-tuning** would mean leaving the backbone unfrozen (and needs a GPU; see Lab 02_03, Part 7).

In [ ]:
hf_model.freeze_base_model()
print(f'trainable parameters: {n_params(hf_model, trainable_only=True):,} of {n_params(hf_model):,}')

hf_forward = lambda m, x: m(input_values=x).logits

torch.manual_seed(0)
np.random.seed(0)                                    # random crops
history = train(hf_model, audio_train, nn.CrossEntropyLoss(weight=class_weights(y_train, 3)),
                epochs=5, lr=1e-3, forward=hf_forward, device=DEVICE)
results_fold0['WavLM + HF head (audio, frozen)'] = uar(*predict(hf_model, audio_test, forward=hf_forward, device=DEVICE))
print('training loss per epoch:', np.round(history, 3))
print(f'fold 0 test UAR = {results_fold0["WavLM + HF head (audio, frozen)"]:.3f}')

## Changing the head of a Hugging Face model
The head is an attribute like any other, so we can replace it.
For MMSE regression we need a single output: replace the `classifier` and use `nn.MSELoss` on the logits.
(We only show the shapes here; training works exactly as above with the MMSE targets.)

In [ ]:
import copy

reg_model = copy.deepcopy(hf_model).cpu()
reg_model.classifier = nn.Linear(reg_model.classifier.in_features, 1)
reg_model.config.num_labels = 1                     # keep the config consistent with the new head
with torch.no_grad():
    pred = reg_model(input_values=xa).logits.squeeze(1)
print('regression output', tuple(pred.shape), '-> loss', nn.MSELoss()(pred, torch.full_like(pred, 26.0)).item())

## Your own model around a Hugging Face backbone
Since everything is an `nn.Module`, we can take only the backbone and build our own model around it: our `LayerWeightedSum`, an **attention pooling** instead of mean pooling, and our own output layer.
Attention pooling learns a score per frame and averages the frames with the softmax of these scores, so the model can focus on the informative parts of the recording.

In [ ]:
class AttentionPooling(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.score = nn.Linear(dim, 1)

    def forward(self, h):                                   # (batch, frames, dim) -> (batch, dim)
        a = torch.softmax(self.score(h), dim=1)             # (batch, frames, 1), sums to 1 over frames
        return (a * h).sum(dim=1)


class MyWavLMClassifier(nn.Module):
    def __init__(self, n_out):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(WAVLM)
        self.backbone.requires_grad_(False)                 # frozen
        cfg = self.backbone.config
        self.layers = LayerWeightedSum(cfg.num_hidden_layers + 1)
        self.pool = AttentionPooling(cfg.hidden_size)
        self.head = nn.Linear(cfg.hidden_size, n_out)

    def train(self, mode=True):
        # frozen is not the same as eval mode: keep dropout and layer drop of the backbone switched off
        super().train(mode)
        self.backbone.eval()
        return self

    def forward(self, x):
        with torch.no_grad():                               # no gradients needed for the frozen backbone
            hidden = torch.stack(self.backbone(x, output_hidden_states=True).hidden_states, dim=1)   # (batch, 13, frames, 768)
        return self.head(self.pool(self.layers(hidden)))


torch.manual_seed(0)
my_model = MyWavLMClassifier(n_out=3)
print(f'trainable parameters: {n_params(my_model, trainable_only=True):,} of {n_params(my_model):,}')

train(my_model, audio_train, nn.CrossEntropyLoss(weight=class_weights(y_train, 3)), epochs=5, device=DEVICE)
results_fold0['WavLM + own attention head (audio, frozen)'] = uar(*predict(my_model, audio_test, device=DEVICE))

pd.Series(results_fold0, name='fold 0 test UAR').round(3)

**Questions**
1. Which lines differ between training the MLP on features (Part 2), the Hugging Face model and our own model? What stayed identical?
2. The Hugging Face head has a `projector` (768 → 256) before pooling. How many parameters does it have compared with our attention head, and what does that mean with about 160 training participants?
3. Why do we not use the `loss` that the Hugging Face model computes?
4. Results on fold 0 with 5 epochs are noisy. What would you need to compare the audio models fairly with the feature models from Part 3?
5. What would you change to fine-tune WavLM instead of training only the head?

**Your turn:** replace `WAVLM` by `'facebook/hubert-base-ls960'` and rerun Part 6. Which lines had to change?

# Part 7: The Hugging Face `Trainer` (15 min)
Our `train` and `predict` functions have about 30 lines.
Hugging Face ships the same loop as the class `Trainer`, plus everything we left out: evaluation and logging during training, checkpoints, learning-rate schedules, mixed precision, gradient accumulation and training on several GPUs.
Most fine-tuning scripts and tutorials on the Hub use it, so you should be able to read and adapt them.

The `Trainer` needs the same ingredients as our loop, under different names:

| our loop | `Trainer` |
|---|---|
| `model` | `model`: any `nn.Module`, typically a Hugging Face model |
| `DataLoader` + collate function | `train_dataset` + `data_collator`; the batch is a **dictionary** whose keys are the argument names of the model (`input_values`, `labels`) |
| `loss_fn` | the loss the model computes itself from `labels`, or our own `compute_loss_func` |
| `epochs`, `lr`, `weight_decay`, `device` | `TrainingArguments` (the device is chosen automatically) |
| `predict` + `uar` | `trainer.predict` / `trainer.evaluate` + `compute_metrics` |

## Data: examples as dictionaries
The `Trainer` uses one collate function for training and evaluation, so the random crop moves from the collate function into the dataset.
Each example is a dictionary with the argument names of the model.
Because all crops have the same length, the default collate function of the `Trainer` can simply stack them.

In [ ]:
from torch.utils.data import Dataset


class AudioCrops(Dataset):
    """Wraps a ParloDataset: example i is a 4 s crop of recording i, as a dictionary of model arguments."""
    def __init__(self, rows, random):
        self.rows, self.random = rows, random

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        row = self.rows[i]
        x = extractor(crop(AUDIO[row['participant_id']], self.random), sampling_rate=SR, return_tensors='pt')
        return {'input_values': x['input_values'][0], 'labels': row['target']}


crops_train, crops_test = AudioCrops(train_set, random=True), AudioCrops(test_set, random=False)
{k: getattr(v, 'shape', v) for k, v in crops_train[0].items()}

## Loss, metric and training arguments
- **Loss:** without further arguments the `Trainer` uses the loss the model computes from `labels` (plain cross-entropy, Part 6).
  With `compute_loss_func` we plug in our class-weighted cross-entropy instead; it receives the model output and the labels.
- **Metric:** `compute_metrics` receives the logits and labels of the whole evaluation set as NumPy arrays and returns a dictionary.
- **Arguments:** we choose the same settings as for our loop in Part 6.
  Note `lr_scheduler_type='constant'`: by default the `Trainer` decreases the learning rate linearly to 0, our loop does not.

In [ ]:
from transformers import Trainer, TrainingArguments

loss_3 = nn.CrossEntropyLoss(weight=class_weights(y_train, 3))


def weighted_ce(outputs, labels, num_items_in_batch=None):
    logits = outputs['logits']                # Hugging Face model outputs can be indexed like dictionaries
    return loss_3.to(logits.device)(logits, labels)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {'uar': balanced_accuracy_score(labels, logits.argmax(axis=1))}


args = TrainingArguments(
    output_dir=str(CACHE_DIR / 'trainer'),    # checkpoints would go here (outside the repository)
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=1e-3,
    weight_decay=1e-2,
    lr_scheduler_type='constant',
    eval_strategy='epoch',                    # evaluate after every epoch ...
    logging_strategy='epoch',                 # ... and log the training loss
    save_strategy='no',                       # no checkpoints for this small experiment
    report_to='none',                         # no external logging service
    dataloader_pin_memory=(DEVICE == 'cuda'), # pinned memory only works with CUDA GPUs
    seed=0,
)

In [ ]:
torch.manual_seed(0)
trainer_model = hf_classifier()
trainer_model.freeze_base_model()

trainer = Trainer(
    model=trainer_model,
    args=args,
    train_dataset=crops_train,
    eval_dataset=crops_test,
    compute_loss_func=weighted_ce,
    compute_metrics=compute_metrics,
)
trainer.train()

`trainer.state.log_history` contains everything that was logged, and `trainer.predict` returns the logits, the labels and the metrics of a data set.

In [ ]:
log = pd.DataFrame(trainer.state.log_history)
display(log.groupby('epoch').first()[['loss', 'eval_loss', 'eval_uar']].round(3))

pred = trainer.predict(crops_test)
results_fold0['WavLM + HF head (audio, frozen), Trainer'] = pred.metrics['test_uar']
print('logits', pred.predictions.shape, '  labels', pred.label_ids.shape)
pd.Series(results_fold0, name='fold 0 test UAR').round(3)

**Questions**
1. Map every argument of `TrainingArguments` above to a line or an argument of our `train` function. Which of them did our loop not have?
2. Why does the UAR of the `Trainer` model differ from the model trained with our loop in Part 6, although model, data and settings are the same?
3. Here `eval_dataset` is the **test** fold. Watching it is fine, but `load_best_model_at_end=True` (keep the epoch with the best `eval_uar`) would make the reported result optimistic. Why? What would you use as `eval_dataset` instead?
4. `compute_loss_func` replaces the loss of the model. What happens if you leave it out?

**Your turn:**
- Leave out `compute_loss_func`, or set `lr_scheduler_type='linear'`, and compare `eval_uar`.
- Train `MyWavLMClassifier` with the `Trainer`. What do you have to change in its `forward` so that the `Trainer` can call it with the dictionary from `AudioCrops`, and so that `weighted_ce` finds the logits? (Hint: argument names and return value; set `label_names=['labels']` in `TrainingArguments`.)

# Part 8: Saving, Reloading and Pipelines (5 min)
`save_pretrained` writes the config and the weights in the Hub format to a **local** folder.
Together with the feature extractor, any Hugging Face tool can then load the model, for example a `pipeline` that does pre-processing, the forward pass and the softmax in one call.
Our own `MyWavLMClassifier` is a plain `nn.Module`: we save its `state_dict` with `torch.save`.

The folder is in the course cache, outside the repository.
`push_to_hub` would upload the folder to the Hub: **never do that with models trained on PARLO.**

In [ ]:
from transformers import pipeline

SAVE_DIR = CACHE_DIR / f'parlo_{TASK}_wavlm_classifier'
hf_model.save_pretrained(SAVE_DIR)
extractor.save_pretrained(SAVE_DIR)
torch.save(my_model.state_dict(), CACHE_DIR / f'parlo_{TASK}_my_wavlm_classifier.pt')
print(sorted(p.name for p in SAVE_DIR.iterdir()))

classify = pipeline('audio-classification', model=str(SAVE_DIR))
row = test_set[0]
print('true:', row['diagnosis'])
classify({'raw': AUDIO[row['participant_id']], 'sampling_rate': SR})

## PARLO as a Hugging Face `Dataset`
The loader can also hand any query to the Hugging Face `datasets` library, which many training scripts expect.
The audio is **not** decoded; the dataset contains the relative audio path, so a `.map` function would load it.
Keep using the participant folds: a random `train_test_split` on recordings would ignore them.

In [ ]:
from parlo.huggingface import to_huggingface

hf_train = to_huggingface(FOLDS[0].apply(query, 'train'))
print(hf_train)
hf_train.filter(lambda r: r['diagnosis'] == 'AD')[:3]

# Summary: What Changes, What Stays the Same
| | stays the same | changes |
|---|---|---|
| data | `Corpus`, `targets`, participant folds, `ParloDataset`, `DataLoader` | the target (`targets('mmse')`, ...) and its dtype; the collate function (features vs. audio) |
| model | body, building blocks (`nn.Linear`, our own modules) | the output layer (`n_out`), possibly a frozen body |
| training | the training loop (ours or the `Trainer`), the optimiser | the loss function; how the model is called (`forward`) |
| evaluation | `predict`, cross-validation over the same folds | the decision rule (argmax, threshold) and the metric (UAR, MAE) |
| Hugging Face | it is all `nn.Module`s | `from_pretrained` / `save_pretrained`, `input_values=...`, `.logits`; `Trainer`: batches as dictionaries, settings in `TrainingArguments` |

**Questions for your PStA**
1. Your audio model and your text model (e.g. BERT via `AutoModel`) can both be wrapped as `nn.Module`s with a common head. Sketch the `forward` of such a hybrid fusion model (Workshop 02_04, Part 8).
2. Which of today's modifications (new head, other loss, layer weights, attention pooling, freezing) would you try first on your data, and how would you evaluate them fairly?